# CS431 · Learned box refinement trên B1 + box-quality v1 · Kaggle T4

Spec khóa trước: `docs/research/janek/refine_spec_20261125_janek.md`.

- **Pipeline:** ảnh → B1 frozen (một forward) → P-map + box → box-quality v1 **giữ nguyên** → MLP nhỏ sửa 4 cạnh của từng box (cùng góc, cùng score, cùng tập box) → evaluator chính thức.
- **Học bằng GT TRAIN** (1.600 nguồn fit; chọn checkpoint trên 400 nguồn calibration). Không B-Clean teacher. Không train detector.
- **A chỉ đánh giá một lần** với checkpoint đã chọn. **Không** mở B/TEST.
- Oracle CPU (dùng GT, chỉ chẩn đoán): sửa cạnh có giới hạn trên A lên Clean 81,28 / mean 80,24 (v1: 79,39 / 78,18). Model phải bắt được khoảng 40–45% lợi ích đó mới qua cổng; **không có gì bảo đảm**.

### Trước khi chạy
1. Kaggle → **New Dataset**, tải `cs431-refine.zip` lên (Private, khoảng 7 MiB).
2. Import notebook này. Bật **GPU T4 x2** (hoặc T4), **Internet On**.
3. **Add Input:** `cs431-refine` **và** dataset `cs431-boxquality-devA` đã có (giữ nguyên, không tải lại).
4. Chạy **Ô 1 → Ô 4**, đọc kết luận Ô 4, rồi mới chạy **Ô 5 → Ô 10**.

**Dừng sau Ô 4 nếu ô in `DỪNG`.** Gửi Claude output Ô 4, không tự đổi cấu hình. Thời gian thật in ở Ô 4 (chưa profile trước).

## Ô 1 · Cấu hình, tìm hai dataset, kiểm GPU

Kiểm SHA của mọi file trong **cả hai** dataset. Dataset devA phải đúng bản đã chạy box-quality (SHA manifest). Ghim **GPU 0**.

In [ ]:
# ===================== Ô 1 · CẤU HÌNH =====================
import hashlib, json, os, shutil, subprocess, sys, time, zipfile
from pathlib import Path

PIN = dict(
    MONKEY_V2_URL='https://github.com/ncat218/MONKEY_V2.git',
    MONKEY_V2_COMMIT='dc19b85c8f73d56dd0510165069597363fe25380',
    MONKEYOCRV2_URL='https://github.com/Yuliang-Liu/MonkeyOCRv2.git',
    MONKEYOCRV2_COMMIT='da96afede92f198c19aca74e1e5f5aa7f4c26af2',
    TORCH='2.9.0', TORCHVISION='0.24.0', TORCHAUDIO='2.9.0', TORCH_INDEX='https://download.pytorch.org/whl/cu128',
    MMENGINE='0.10.7', MMCV='2.0.1', MMOCR='1.0.1', PILLOW='11.3.0', OPENCV='4.10.0.84',
    DB_PP_SHA256='9bfff8aaaca103dd08d7565f2b7a5410ab20a6427fe1f0baad5a5d04d207c8b8',
    BUNDLE_MANIFEST_SHA256='e06ef54e720b43bcb658fd734291fb1b6e4a08db13a8bd4048f16cfd3a2dc573',
)
W = Path('/kaggle/working'); OUT = W/'cs431_out'; LOGS = OUT/'logs'
T = Path('/tmp/cs431')
for p in (OUT, LOGS, T):
    p.mkdir(parents=True, exist_ok=True)

def fail(msg):
    raise SystemExit('LỖI: ' + msg)

def sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(1 << 20), b''):
            h.update(block)
    return h.hexdigest()

def find(kind, hint):
    found = [p for p in Path('/kaggle/input').glob('**/KAGGLE_MANIFEST.json') if json.loads(p.read_text()).get('kind') == kind]
    if not found:
        for z in [z for z in Path('/kaggle/input').glob('**/*.zip') if hint in z.name.lower()]:
            with zipfile.ZipFile(z) as zf:
                zf.extractall(T/f'ds_{kind}')
        found = [p for p in T.glob(f'ds_{kind}/**/KAGGLE_MANIFEST.json') if json.loads(p.read_text()).get('kind') == kind]
    if len(found) != 1:
        fail(f'Thấy {len(found)} dataset kind={kind}; cần đúng một. Bấm "Add Input".')
    m = json.loads(found[0].read_text())
    bad = [rel for rel, info in m['files'].items() if not (found[0].parent/rel).is_file() or sha256(found[0].parent/rel) != info['sha256']]
    if bad:
        fail(f'Dataset {kind}: {len(bad)} file thiếu/sai SHA, ví dụ {bad[:5]}')
    print(f'Dataset {kind}: {found[0].parent} ({len(m["files"])} file, SHA khớp)')
    return found[0].parent, m, sha256(found[0])

RD, RMAN, _ = find('refine', 'refine')
DS, DMAN, DMAN_SHA = find('boxquality_devA', 'deva')
if DMAN_SHA != RMAN['devA_dataset_manifest_sha256']:
    fail('Dataset cs431-boxquality-devA khác bản đã chạy box-quality (SHA manifest).')

CODE = T/'code'
if CODE.exists():
    shutil.rmtree(CODE)
shutil.copytree(RD/'code', CODE)
code_sha = hashlib.sha256((RD/'code/CODE_MANIFEST.json').read_bytes()).hexdigest()
if str(CODE) not in sys.path:
    sys.path.insert(0, str(CODE))
import importlib
import mixab.colab_helpers_janek as ch
importlib.reload(ch)

gpus = (ch.output_of(['nvidia-smi', '--query-gpu=index,name,driver_version,memory.total', '--format=csv,noheader']) or '').splitlines()
if not gpus:
    fail('Không thấy GPU. Settings -> Accelerator -> GPU T4 x2 (hoặc GPU T4).')
print('GPU của máy:'); [print('  ', g) for g in gpus]
GPU = ch.gpu_label(gpus[0].split(',')[1].strip())
CONFIG_NOTE = f'{len(gpus)} GPU trên máy; chỉ dùng GPU 0 qua CUDA_VISIBLE_DEVICES=0'
print(CONFIG_NOTE)

PY = sys.executable
MV2, MOCR = T/'MONKEY_V2', T/'MonkeyOCRv2'
BUNDLE = MV2/'docs/handoff/artifacts/pr_quad_audit_v1'
WEIGHTS, CLEAN_A = DS/'weights', DS/'clean'
MODEL, FIT = DS/'model/model_janek.npz', DS/'model/boxquality_fit_janek.json'
ENCODER = T/'encoder'
if not (ENCODER/'model.safetensors').is_file():
    shutil.copytree(DS/'encoder', ENCODER, dirs_exist_ok=True)
PREP_A, PREP_T, TRAIN_CLEAN, CONFIG_PATH = T/'prepared_A', T/'prepared_T', T/'train_clean', T/'inference_config_kaggle.py'
INP = T/'inputs'
ENV = dict(MONKEY_V2_ROOT=str(MV2), PYTHONPATH=os.pathsep.join((str(CODE), str(MV2))), CUDA_VISIBLE_DEVICES='0')
REF = RD/'ref/devA_result_janek.json'
print(f'Python {sys.version.split()[0]} · code manifest {code_sha[:12]}… · kết quả: {OUT}')

## Ô 2 · Cài môi trường (cùng phiên bản với lần chạy dump T4 trên Colab)

Thứ tự: torch 2.9.0 cu128 → mmengine 0.10.7 → **MMCV 2.0.1** (dùng wheel trong dataset nếu cùng phiên bản Python, nếu không thì **build từ source**) → `install.sh` của MonkeyOCRv2 (MMOCR 1.0.1 + add-on) → ghim Pillow 11.3.0 và OpenCV 4.10.0.84.
- Build MMCV có thể mất **20–40 phút** (ước lượng, chưa đo trên Kaggle). Wheel build xong được chép vào `/kaggle/working/cs431_out/wheels/`: **tải về và thêm vào dataset** để lần sau khỏi build.
- Cần `nvcc` CUDA **12.x** để build. Ô này dùng toolkit có sẵn trên Kaggle (ưu tiên 12.8); nếu không có thì cài `cuda-toolkit-12-8` (không cài driver).
- Kiểm cuối chạy trong tiến trình mới: phiên bản, `MonkeyOCRv2ViTAEBackbone` đã đăng ký, SHA của DBPostprocessor phải đúng bản Tuấn ghim.
- Nếu Kaggle báo cần *Restart*, restart rồi chạy lại **Ô 1** và **Ô 2** (phần đã cài sẽ bỏ qua).

In [ ]:
# ===================== Ô 2 · CÀI MÔI TRƯỜNG =====================
import platform, re
LOG = LOGS/f'install_{GPU}.log'
pyver = sys.version_info
if not ((3, 10) <= pyver[:2] <= (3, 13)):
    print(f'CẢNH BÁO: Python {platform.python_version()} chưa được kiểm với MMCV 2.0.1.')

def ver(name):
    return ch.package_version(name)

def pip(*args):
    ch.run([PY, '-m', 'pip', 'install', '-q', *args], LOG, title='pip install ' + ' '.join(args))

if ver('torch') != f"{PIN['TORCH']}+cu128" or ver('torchvision') != f"{PIN['TORCHVISION']}+cu128":
    pip(f"torch=={PIN['TORCH']}", f"torchvision=={PIN['TORCHVISION']}", '--index-url', PIN['TORCH_INDEX'])
if ver('torchaudio') != f"{PIN['TORCHAUDIO']}+cu128":
    pip('--no-deps', f"torchaudio=={PIN['TORCHAUDIO']}+cu128", '--index-url', PIN['TORCH_INDEX'])
cuda_ok = ch.output_of([PY, '-c', 'import torch;print(torch.version.cuda, torch.cuda.is_available())'], env=ENV)
print('torch CUDA:', cuda_ok)
if not cuda_ok or 'True' not in cuda_ok:
    fail('torch không dùng được CUDA (driver Kaggle quá cũ cho cu128?). Gửi log: ' + str(LOG))
if ver('mmengine') != PIN['MMENGINE']:
    pip(f"mmengine=={PIN['MMENGINE']}")
pip('setuptools<81', 'wheel', 'ninja')

def mmcv_ok():
    if ver('mmcv') != PIN['MMCV']:
        return False
    probe = subprocess.run([PY, '-c', 'from mmcv.ops import nms; print("IMPORT_NMS_OK")'], capture_output=True, text=True)
    with LOG.open('a', encoding='utf-8') as log:
        log.write('\nMMCV import probe:\n' + probe.stdout + probe.stderr)
    return probe.returncode == 0 and 'IMPORT_NMS_OK' in probe.stdout.splitlines()

def find_toolkit():
    """CUDA 12.x toolkit with nvcc + cuda.h; prefer 12.8. torch only refuses a different MAJOR version."""
    import glob
    homes = ['/usr/local/cuda-12.8', os.environ.get('CUDA_HOME'), '/usr/local/cuda', *sorted(glob.glob('/usr/local/cuda-12*'))]
    nvcc = shutil.which('nvcc')
    if nvcc:
        homes.append(str(Path(nvcc).resolve().parents[1]))
    best = None
    for home in homes:
        if not home or not (Path(home)/'bin/nvcc').is_file() or not (Path(home)/'include/cuda.h').is_file():
            continue
        m = re.search(r'release (\d+)\.(\d+)', ch.output_of([str(Path(home)/'bin/nvcc'), '--version']) or '')
        if m and m.group(1) == '12':
            cand = (int(m.group(2)) == 8, int(m.group(2)), str(Path(home).resolve()))
            best = max(best, cand) if best else cand
    return best

tag = f'cp{pyver[0]}{pyver[1]}'
if not mmcv_ok():
    wheels = sorted((DS/'wheels').glob(f"mmcv-{PIN['MMCV']}-{tag}-*.whl")) + sorted(OUT.glob(f"wheels/mmcv-{PIN['MMCV']}-{tag}-*.whl"))
    if wheels:
        print('Cài MMCV từ wheel có sẵn:', wheels[0].name)
        pip('--no-deps', str(wheels[0]))
    else:
        tk = find_toolkit()
        if not tk:
            print('Không có CUDA toolkit 12.x. Cài cuda-toolkit-12-8 từ NVIDIA (không cài driver)...')
            distro = platform.freedesktop_os_release()
            repo = 'ubuntu' + distro.get('VERSION_ID', '22.04').replace('.', '')
            from urllib.request import urlretrieve
            keyring = T/'cuda-keyring_1.1-1_all.deb'
            urlretrieve(f'https://developer.download.nvidia.com/compute/cuda/repos/{repo}/x86_64/cuda-keyring_1.1-1_all.deb', keyring)
            ch.run(['dpkg', '-i', keyring], LOG)
            ch.run(['apt-get', 'update'], LOG)
            ch.run(['apt-get', 'install', '-y', '--no-install-recommends', 'cuda-toolkit-12-8'], LOG, env={'DEBIAN_FRONTEND': 'noninteractive'})
            tk = find_toolkit()
            if not tk:
                fail('Đã cài nhưng vẫn không thấy nvcc 12.x; xem ' + str(LOG))
        cuda_home = tk[2]
        print(f'Dùng CUDA toolkit {cuda_home} (12.{tk[1]})' + ('' if tk[0] else ' — khác 12.8, torch chỉ cảnh báo khác minor version'))
        src = T/'mmcv_src'
        if not (src/'setup.py').is_file():
            ch.run(['git', 'clone', '-b', f"v{PIN['MMCV']}", '--depth', '1', 'https://github.com/open-mmlab/mmcv.git', src], LOG)
        setup = (src/'setup.py').read_text()
        setup = setup.replace('-std=c++14', '-std=c++17')
        old_v = "        exec(compile(f.read(), version_file, 'exec'))\n    return locals()['__version__']"
        new_v = "        namespace = {}\n        exec(compile(f.read(), version_file, 'exec'), namespace)\n    return namespace['__version__']"
        if old_v in setup:
            setup = setup.replace(old_v, new_v)
        elif new_v not in setup:
            fail('MMCV setup.py khác cấu trúc đã kiểm.')
        (src/'setup.py').write_text(setup)
        print(f'Build MMCV ({os.cpu_count()} CPU). Có thể mất lâu...')
        build_env = dict(CUDA_HOME=cuda_home, MMCV_WITH_OPS='1', MAX_JOBS=str(os.cpu_count() or 2),
                         PATH=f"{cuda_home}/bin:{os.environ.get('PATH', '')}", TORCH_CUDA_ARCH_LIST='7.5;8.0')
        wheel_tmp = T/'wheelhouse'
        ch.run([PY, '-m', 'pip', 'wheel', '-v', '--no-build-isolation', '--no-deps', '-w', wheel_tmp, src], LOG,
               env=build_env, title='build MMCV wheel')
        built = sorted(wheel_tmp.glob('mmcv-*.whl'))
        if not built:
            fail('Build MMCV không tạo ra wheel; xem ' + str(LOG))
        (OUT/'wheels').mkdir(exist_ok=True)
        shutil.copy2(built[-1], OUT/'wheels'/built[-1].name)
        print('Đã lưu wheel để tải về:', OUT/'wheels'/built[-1].name)
        pip('--no-deps', str(built[-1]))
    if not mmcv_ok():
        fail('MMCV chưa import được mmcv.ops.nms; xem ' + str(LOG))

if not (MOCR/'.git').is_dir():
    ch.run(['git', 'clone', PIN['MONKEYOCRV2_URL'], MOCR], LOG)
ch.run(['git', '-C', MOCR, 'checkout', '--detach', PIN['MONKEYOCRV2_COMMIT']], LOG)
addon = MOCR/'detection/mmocr'
mmocr_root = addon/'mmocr'
backbone_file = mmocr_root/'mmocr/models/common/backbones/monkeyocr_v2_vitae.py'
mmocr_loc = ch.output_of([PY, '-c', 'import mmocr,os;print(os.path.dirname(mmocr.__file__))'])
if not (ver('mmocr') == PIN['MMOCR'] and backbone_file.is_file() and mmocr_loc and str(mmocr_root) in mmocr_loc):
    if not (mmocr_root/'.git').is_dir():
        ch.run(['git', 'clone', '--branch', 'v1.0.1', '--depth', '1', 'https://github.com/open-mmlab/mmocr.git', mmocr_root], LOG)
    setup_path = mmocr_root/'setup.py'
    setup = setup_path.read_text()
    old_exec = "        exec(compile(f.read(), version_file, 'exec'))"
    new_exec = "        namespace = {}\n        exec(compile(f.read(), version_file, 'exec'), namespace)"
    if old_exec in setup:
        setup = setup.replace(old_exec, new_exec)
        setup = setup.replace("locals()['short_version']", "namespace['short_version']").replace("locals()['__version__']", "namespace['__version__']")
        setup_path.write_text(setup)
    elif new_exec not in setup:
        fail('MMOCR setup.py khác cấu trúc đã kiểm.')
    ch.run(['bash', 'install.sh'], LOG, cwd=addon, title='MonkeyOCRv2 detection/mmocr/install.sh',
           env=dict(PATH=f"{Path(PY).parent}:{os.environ.get('PATH', '')}", PIP_VERBOSE='1'))
if ver('Pillow') != PIN['PILLOW']:
    pip(f"Pillow=={PIN['PILLOW']}")
if ver('opencv-python') != PIN['OPENCV'] or any(ver(n) for n in ('opencv-python-headless', 'opencv-contrib-python')):
    ch.run([PY, '-m', 'pip', 'uninstall', '-y', 'opencv-python-headless', 'opencv-contrib-python', 'opencv-contrib-python-headless'], LOG)
    pip('--force-reinstall', '--no-deps', f"opencv-python=={PIN['OPENCV']}")
if ver('shapely') is None:
    pip('shapely')

VERIFY = """
import hashlib, importlib.metadata as m, importlib.util, json, torch
from mmocr.utils import register_all_modules
from mmocr.registry import MODELS
from mmcv.ops import nms
register_all_modules(init_default_scope=True)
spec = importlib.util.find_spec('mmocr.models.textdet.postprocessors.db_postprocessor')
out = dict(versions={k: m.version(k) for k in ('torch','torchvision','mmengine','mmcv','mmdet','mmocr','Pillow','opencv-python','numpy','shapely')},
           cuda=torch.cuda.is_available(), device_count_visible=torch.cuda.device_count(), device=torch.cuda.get_device_name(0),
           backbone_registered=MODELS.get('MonkeyOCRv2ViTAEBackbone') is not None,
           db_postprocessor_sha256=hashlib.sha256(open(spec.origin,'rb').read()).hexdigest())
print(json.dumps(out))
"""
raw = ch.output_of([PY, '-c', VERIFY], env=ENV)
if not raw:
    fail('Kiểm cuối môi trường lỗi khi import; xem ' + str(LOG))
env_report = json.loads(raw.splitlines()[-1])
(OUT/f'env_{GPU}.json').write_text(json.dumps(dict(env_report, pins=PIN, gpus=gpus, config=CONFIG_NOTE), indent=1))
print(json.dumps(env_report, indent=1))
problems = []
if env_report['db_postprocessor_sha256'] != PIN['DB_PP_SHA256']:
    problems.append('DBPostprocessor khác bản Tuấn ghim')
if not env_report['backbone_registered']:
    problems.append('MonkeyOCRv2ViTAEBackbone chưa đăng ký')
if env_report['versions']['Pillow'] != PIN['PILLOW']:
    problems.append('Pillow không phải 11.3.0 (cần để tạo LR2 đúng byte)')
if env_report['device_count_visible'] != 1:
    problems.append('Tiến trình đo thấy nhiều hơn 1 GPU')
if problems:
    fail('Môi trường chưa đạt: ' + '; '.join(problems))
print('\nÔ 2 xong.')

## Ô 3 · Repo của Tuấn, config, B1, scorer đóng băng, ảnh A, nhãn TRAIN, ảnh smoke TRAIN

- Clone `MONKEY_V2` @ `dc19b85`, kiểm `bundle_manifest.json`, vá config (chỉ đường dẫn encoder), kiểm SHA B1.
- Scorer v1 đóng băng (kiểm SHA trong `load_model`).
- A: kiểm SHA 1.130 ảnh Clean, dựng 1.130 LR2 và kiểm SHA task đã khóa.
- TRAIN: khôi phục `train_rows` (kiểm SHA), tải 10 ảnh **smoke** TRAIN (CRC + SHA), dựng LR2. Chạy test CPU.

In [ ]:
# ===================== Ô 3 · DỮ LIỆU =====================
import csv
from pathlib import PureWindowsPath
import mixab.boxquality_devA_janek as dv, mixab.boxquality_janek as bq
importlib.reload(dv); importlib.reload(bq)
LOG = LOGS/f'data_{GPU}.log'
if not (MV2/'.git').is_dir():
    ch.run(['git', 'clone', PIN['MONKEY_V2_URL'], MV2], LOG, env=dict(GIT_LFS_SKIP_SMUDGE='1'))
ch.run(['git', '-C', MV2, 'checkout', '--detach', PIN['MONKEY_V2_COMMIT']], LOG, env=dict(GIT_LFS_SKIP_SMUDGE='1'))
if ch.output_of(['git', '-C', MV2, 'status', '--porcelain', '--untracked-files=no']):
    fail('Checkout MONKEY_V2 bị sửa.')
if sha256(BUNDLE/'bundle_manifest.json') != PIN['BUNDLE_MANIFEST_SHA256']:
    fail('bundle_manifest.json khác bản đã khóa.')
CONFIG_PATH.write_text(ch.patch_inference_config((MV2/'docs/handoff/inference_config.py').read_text(encoding='utf-8'), ENCODER), encoding='utf-8')
m = json.loads((WEIGHTS/'merge_manifest.json').read_text())
b1 = next(x for x in m['models'] if x['alpha'] == '1')
if sha256(WEIGHTS/PureWindowsPath(b1['checkpoint']).name) != b1['checkpoint_sha256']:
    fail('alpha_1p00.pth (B1) sai SHA.')
model = dv.load_model(MODEL, FIT)
print(f"Scorer v1 đóng băng: cut_full {model['cut_full']!r} · {len(model['features'])} đặc trưng")
ch.run([PY, '-m', 'mixab.boxquality_devA_janek', 'prepare', '--bundle', BUNDLE, '--clean-root', CLEAN_A, '--prepared-root', PREP_A,
        '--subset', 'all'], LOG, cwd=CODE, env=ENV, title='A: check Clean SHA, build + check LR2')
INP.mkdir(parents=True, exist_ok=True)
shutil.copyfile(RD/'inputs/train_rows.gzip.bin', INP/'train_rows.jsonl.gz')
for f_ in ('selection.csv', 'train_ids.csv'):
    shutil.copyfile(RD/'inputs'/f_, INP/f_)
if sha256(INP/'train_rows.jsonl.gz') != RMAN['train_rows_sha256'] or sha256(INP/'selection.csv') != RMAN['files']['inputs/selection.csv']['sha256']:
    fail('Nhãn TRAIN hoặc selection sai SHA.')
ROWS = INP/'train_rows.jsonl.gz'
SEL = list(csv.DictReader(open(INP/'selection.csv', encoding='utf-8')))
r_ = bq.fetch_members([(x['clean_relpath'], x['clean_sha256']) for x in SEL if x['role'] == 'smoke'], TRAIN_CLEAN)
print('Ảnh smoke TRAIN:', r_)
ch.run([PY, '-m', 'mixab.boxquality_janek', 'prepare', '--rows', ROWS, '--clean-root', TRAIN_CLEAN, '--prepared-root', PREP_T,
        '--roles', 'smoke'], LOG, cwd=CODE, env=ENV, title='prepare smoke LR2')
ch.run([PY, '-m', 'unittest', 'discover', '-s', 'tests', '-t', '.'], LOG, cwd=CODE, env=ENV, title='mixab unit tests')
print('\nÔ 3 xong.')

## Ô 4 · Smoke: 10 nguồn smoke TRAIN (20 task) + 5 nguồn A (10 task), kiểm ở MỌI task

1. **Crop không đổi output:** mỗi task chạy thêm `bq.run_task` (forward thứ hai) và phải cho record v1 **trùng tuyệt đối**; post-process phải trùng `capture_candidates` của Tuấn.
2. **Tọa độ:** quad mạng tính lại từ polygon cuối (qua restore + scale_factor) phải trùng quad của post-processor (< 0,01 px), và nằm trong ảnh (lệch ≤ 2 px).
3. In thời gian thật mỗi giai đoạn và dự kiến cho 4.000 task TRAIN + 2.260 task A.

**Dừng** nếu một kiểm sai, hoặc dự kiến > 4 giờ. Số Hmean không in ở ô này.

In [ ]:
# ===================== Ô 4 · SMOKE =====================
EXT = lambda split, out, extra: [PY, '-m', 'mixab.refine_cmd_janek', 'extract', '--split', split, '--model-config', CONFIG_PATH,
                                 '--weights-dir', WEIGHTS, '--output', out, *extra]
ST = OUT/'smoke_train'
if not (ST/'extract_summary.json').is_file():
    ch.run(EXT('train', ST, ['--rows', ROWS, '--roles', 'smoke', '--clean-root', TRAIN_CLEAN, '--prepared-root', PREP_T,
                            '--consistency', '--shard', '20']), LOGS/'smoke_train.log', cwd=CODE, env=ENV, title='smoke TRAIN extract')
SA = OUT/'smoke_A'
if not (SA/'extract_summary.json').is_file():
    ch.run(EXT('devA', SA, ['--bundle', BUNDLE, '--clean-root', CLEAN_A, '--prepared-root', PREP_A, '--limit', '5',
                           '--consistency', '--shard', '10']), LOGS/'smoke_A.log', cwd=CODE, env=ENV, title='smoke A extract')
ok, per_task = True, []
for name, d in (('TRAIN smoke', ST), ('A smoke', SA)):
    s = json.loads((d/'extract_summary.json').read_text())
    # Read per-task evidence, including summaries created by the older dataset.
    records = bq.load_records(d)
    s['consistency_all_true'] = (len(records) == s['records'] and len(records) > 0
                                 and all(r.get('consistency') is True for r in records))
    print(f"{name}: {s['records']} task · {s['candidates']} ứng viên · kiểm {s['n_checks']} task, trùng v1 = {s['checks_all_identical']}, "
          f"post-process = capture_candidates: {s['consistency_all_true']} · lệch quad mạng {s['max_net_quad_dev_px']:.2e} px · "
          f"ngoài ảnh {s['max_outside_px']:.2f} px · VRAM {s['peak_gpu_mib']:.0f} MiB")
    print('   ms median:', {k_: round(v['median'], 1) for k_, v in s['ms'].items()})
    ok &= s['checks_all_identical'] and s['consistency_all_true'] and s['n_checks'] == s['records'] and s['max_outside_px'] <= 2
    per_task.append(sum(v['median'] for v in s['ms'].values())/1000 + .05)
sec = max(per_task)
proj = sec*(4000*(1 + 1/25) + 2260*(1 + 1/25))/60
print(f'\nDự kiến trích TRAIN 4.000 + A 2.260 task: khoảng {proj:.0f} phút (từ median đo thật + 50 ms đọc ảnh/task; chưa gồm tải 2.000 ảnh TRAIN)')
if not ok:
    print('DỪNG: có kiểm hợp lệ sai. Gửi Claude output ô này.')
elif proj > 240:
    print('DỪNG: dự kiến vượt 4 giờ. Báo Claude.')
else:
    print('Được chạy Ô 5.')

## Ô 5 · Tải 2.000 ảnh TRAIN, dựng LR2, trích 4.000 task TRAIN (GPU)

- Ảnh tải từng file (CRC + SHA-256); ảnh đã có thì bỏ qua.
- Lưu sau mỗi 50 task (`cs431_out/extract_train/`, gồm `.jsonl.gz` + `.npz` crop). Bị ngắt **trong cùng phiên** thì chạy lại ô này.
- Cứ 25 task kiểm lại một lần như Ô 4.

In [ ]:
# ===================== Ô 5 · TRÍCH TRAIN =====================
tic = time.time()
r_ = bq.fetch_members([(x['clean_relpath'], x['clean_sha256']) for x in SEL if x['role'] in ('fit', 'calibration')], TRAIN_CLEAN)
print(f"Ảnh fit + calibration: {r_} ({(time.time() - tic)/60:.1f} phút)")
ch.run([PY, '-m', 'mixab.boxquality_janek', 'prepare', '--rows', ROWS, '--clean-root', TRAIN_CLEAN, '--prepared-root', PREP_T,
        '--roles', 'fit,calibration'], LOGS/'prepare_train.log', cwd=CODE, env=ENV, title='prepare TRAIN LR2')
ET = OUT/'extract_train'
ch.run(EXT('train', ET, ['--rows', ROWS, '--roles', 'fit,calibration', '--clean-root', TRAIN_CLEAN, '--prepared-root', PREP_T]),
       LOGS/f'extract_train_{time.strftime("%Y%m%d-%H%M%S")}.log', cwd=CODE, env=ENV, title='TRAIN extract')
s = json.loads((ET/'extract_summary.json').read_text())
print({k_: s[k_] for k_ in ('tasks', 'records', 'candidates', 'n_checks', 'checks_all_identical', 'max_net_quad_dev_px', 'ms', 'peak_gpu_mib')})
if not (s['records'] == 4000 and s['checks_all_identical']):
    print('DỪNG: chưa đủ 4.000 task hoặc có kiểm sai. Chạy lại ô (resume) hoặc gửi Claude.')

## Ô 6 · Trích split A (2.260 task, GPU)

Chỉ trích (P-map crop + record v1). **Chưa** đánh giá; A được dùng đúng một lần ở Ô 8 sau khi đã chọn checkpoint trên TRAIN.

In [ ]:
# ===================== Ô 6 · TRÍCH A =====================
EA = OUT/'extract_A'
ch.run(EXT('devA', EA, ['--bundle', BUNDLE, '--clean-root', CLEAN_A, '--prepared-root', PREP_A]),
       LOGS/f'extract_A_{time.strftime("%Y%m%d-%H%M%S")}.log', cwd=CODE, env=ENV, title='A extract')
s = json.loads((EA/'extract_summary.json').read_text())
print({k_: s[k_] for k_ in ('tasks', 'records', 'candidates', 'n_checks', 'checks_all_identical', 'max_net_quad_dev_px', 'ms', 'peak_gpu_mib')})
if not (s['records'] == 2260 and s['checks_all_identical']):
    print('DỪNG: chưa đủ 2.260 task hoặc có kiểm sai. Chạy lại ô (resume) hoặc gửi Claude.')

## Ô 7 · Train bộ sửa (CPU) + chọn checkpoint trên TRAIN-calibration

- Train trên fold **fit** (cặp box v1 ↔ GT ghép một-một; detector frozen). Công thức khóa trong `mixab/refine_janek.py`.
- Chọn trong e05…e30 theo mean Hmean chính thức trên 400 nguồn **calibration**; phải hơn v1 (offset 0). Không thì `NO_SIGNAL_ON_CALIBRATION` và **không** chạy Ô 8.

In [ ]:
# ===================== Ô 7 · TRAIN + CHỌN =====================
TR = OUT/'train'
COMMON = ['--bundle', BUNDLE, '--model', MODEL, '--fit', FIT]
if not (TR/'train.json').is_file():
    ch.run([PY, '-m', 'mixab.refine_cmd_janek', 'train', *COMMON, '--rows', ROWS, '--extract-dir', ET, '--output', TR],
           LOGS/'train.log', cwd=CODE, env=ENV, title='train refiner (TRAIN fit)')
t_ = json.loads((TR/'train.json').read_text())
print(f"cặp train {t_['pairs']} (TP {t_['tp_pairs']}) · {t_['seconds']:.0f} s · loss", [round(h['loss'], 5) for h in t_['history'][::5]])
SELJ = OUT/'selection.json'
if not SELJ.is_file():
    ch.run([PY, '-m', 'mixab.refine_cmd_janek', 'select', *COMMON, '--rows', ROWS, '--extract-dir', ET, '--train-dir', TR,
            '--output', SELJ], LOGS/'select.log', cwd=CODE, env=ENV, title='select on TRAIN-calibration')
S_ = json.loads(SELJ.read_text())
for n_, v_ in S_['table'].items():
    print(f"  {n_:4} mean {v_['mean']:.3f} · Clean {v_['clean']:.3f} · LR2 {v_['lr2']:.3f}")
print('Chọn:', S_['chosen'], S_['chosen_minus_v1'], '·', S_['outcome'])
if S_['chosen']:
    d_ = S_['diag'][S_['chosen']]
    print('  GT cứu/mất:', d_['gained'], '/', d_['lost'], '· fallback:', d_['status'])
    print('  độ dịch theo loại box (tỉ lệ > 0,10 cạnh):', {k_: round(v_['over_0_10']/max(v_['n'], 1), 3) for k_, v_ in d_['moved'].items()})
else:
    print('DỪNG: không có tín hiệu trên calibration. Không chạy Ô 8. Chạy Ô 10 để đóng gói, gửi Claude.')

## Ô 8 · Một lượt đánh giá A đóng băng (CPU)

F (B1 `.40/.60`), v1, v1 + refinement. Kiểm F và v1 trùng `devA_result`. Nhãn khóa theo spec §5. **Không chỉnh gì sau khi xem.**

In [ ]:
# ===================== Ô 8 · ĐÁNH GIÁ A =====================
EV = OUT/'eval_A'
if not (EV/'refine_devA.json').is_file():
    ch.run([PY, '-m', 'mixab.refine_cmd_janek', 'evaluate', *COMMON, '--extract-dir', EA, '--train-dir', TR, '--selection', SELJ,
            '--devA-ref', REF, '--output', EV], LOGS/'evaluate.log', cwd=CODE, env=ENV, title='evaluate A (once)')
R_ = json.loads((EV/'refine_devA.json').read_text())
print('Trùng devA_result (F, v1):', R_['reproduces_devA_result'])
for n_ in ('F', 'v1', 'v1R'):
    a_ = R_['arms'][n_]
    print(f"{n_:4} Clean {a_['clean']['hmean']*100:.2f} (TP {a_['clean']['tp']} FP {a_['clean']['fp']} FN {a_['clean']['fn']}, tiny {a_['clean']['tiny_recall']*100:.1f}) · "
          f"LR2 {a_['lr2']['hmean']*100:.2f} (TP {a_['lr2']['tp']} FP {a_['lr2']['fp']} FN {a_['lr2']['fn']}, tiny {a_['lr2']['tiny_recall']*100:.1f}) · mean {a_['mean_hmean']*100:.2f}")
for k_, d_ in R_['differences'].items():
    print(f"{k_:7} mean {d_['mean']['delta_pp']:+.2f} [{d_['mean']['ci95_pp'][0]:+.2f}; {d_['mean']['ci95_pp'][1]:+.2f}] · Clean {d_['clean']['delta_pp']:+.2f} · LR2 {d_['lr2']['delta_pp']:+.2f}")
print('GT cứu/mất:', R_['diag']['gained'], '/', R_['diag']['lost'], '· fallback:', R_['diag']['status'])
print('Cổng:', R_['gates'], R_['gate_values'])
print('\nNHÃN:', R_['outcome'])

## Ô 9 · Latency + bộ nhớ (GPU)

100 nguồn A × Clean/LR2, 2 lượt, thứ tự 3 cánh xoay vòng. Phần sửa gồm: chuyển P-map về CPU, crop, MLP (CPU), dựng quad. **Không chạy việc khác** trong lúc đo.

In [ ]:
# ===================== Ô 9 · LATENCY =====================
LT = OUT/'latency'
if S_['chosen'] and not (LT/'latency.json').is_file():
    ch.run([PY, '-m', 'mixab.refine_cmd_janek', 'latency', *COMMON, '--clean-root', CLEAN_A, '--prepared-root', PREP_A,
            '--model-config', CONFIG_PATH, '--weights-dir', WEIGHTS, '--train-dir', TR, '--selection', SELJ, '--output', LT],
           LOGS/'latency.log', cwd=CODE, env=ENV, title='latency F / v1 / v1R')
if (LT/'latency.json').is_file():
    L_ = json.loads((LT/'latency.json').read_text())
    for n_, v_ in L_['ms'].items():
        print(f"{n_:4} median {v_['total']['median']:.1f} ms · p95 {v_['total']['p95']:.1f} · phần sửa median {v_['refine']['median']:.1f} ms · VRAM {L_['memory'][n_]['peak_allocated_mib']:.0f} MiB")
    print('v1R/v1 median', round(L_['v1R_over_v1']['ratio_of_medians'], 3), L_['v1R_over_v1']['ratio_of_medians_ci95'],
          '· v1R/F', round(L_['v1R_over_F']['ratio_of_medians'], 3))

## Ô 10 · Đóng gói

Tạo `/kaggle/working/cs431_refine_<GPU>_<thời gian>.zip` gồm toàn bộ `cs431_out` (bản ghi, crop, checkpoint, kết quả, log). Tải ở khung **Output** và gửi Claude (hoặc báo Claude tải bằng Kaggle API).

In [ ]:
# ===================== Ô 10 · ĐÓNG GÓI =====================
pack = W/f'cs431_refine_{GPU}_{time.strftime("%Y%m%d-%H%M%S")}.zip'
with zipfile.ZipFile(pack, 'w', compression=zipfile.ZIP_STORED) as zf:
    for p in sorted(OUT.rglob('*')):
        if p.is_file() and p.suffix != '.whl':
            zf.write(p, p.relative_to(OUT).as_posix())
    zf.writestr('code_manifest_sha256.txt', code_sha + '\n')
print('Đã đóng gói:', pack, f'({pack.stat().st_size/2**20:.1f} MiB)')